# Pipeline de Optimización ARIMA para The Coca-Cola Company (KO)
El cuaderno ejecuta el análisis econométrico y la búsqueda en cuadrícula (*Grid Search*) para determinar los componentes óptimos $(p, d, q)$ del modelo ARIMA aplicados a la serie temporal de Coca-Cola, sincronizado con los resultados del reporte estilo NeurIPS.

In [ ]:
import logging
import warnings
import pandas as pd
import yfinance as yf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
warnings.filterwarnings("ignore")
print("Librerías importadas y entorno configurado con éxito.")

Librerías importadas y entorno configurado con éxito.


In [ ]:
def descargar_datos(ticker: str, start_date: str, end_date: str) -> pd.Series:
    """Descarga precios de cierre manejando estructuras MultiIndex de yfinance."""
    try:
        df = yf.download(ticker, start=start_date, end=end_date, group_by='ticker')
        if df.empty:
            raise ValueError(f"No se encontraron datos para el ticker {ticker}.")
        
        if isinstance(df.columns, pd.MultiIndex):
            serie = df[ticker]['Close'].dropna() if 'Close' in df[ticker].columns else df[ticker]['Adj Close'].dropna()
        else:
            serie = df['Close'].dropna() if 'Close' in df.columns else df['Adj Close'].dropna()
            
        return pd.Series(serie.values.flatten(), index=serie.index)
    except Exception as e:
        print(f"Error en la descarga: {e}")
        raise

TICKER = "KO"
FECHA_INICIO = "2025-01-01"
FECHA_FIN = "2026-01-01"

serie_precios = descargar_datos(TICKER, FECHA_INICIO, FECHA_FIN)

[*********************100%***********************]  1 of 1 completed


## 1. Análisis de Estacionariedad
Para determinar el componente de integración ($d$), aplicamos la prueba formal de **Dickey-Fuller Aumentada (ADF)** tanto a la serie original en niveles como a su primera diferencia.

In [6]:
print("--- PRUEBAS DE DICKEY-FULLER AUMENTADA (ADF) ---")

# ADF en Niveles (d=0)
adf_niveles = adfuller(serie_precios)
print(f"1. Serie en Niveles (d=0): Estadístico = {adf_niveles[0]:.4f} | p-valor = {adf_niveles[1]:.4f}")
print("   Veredicto: No se rechaza H0. Presencia de Raíz Unitaria (No Estacionaria).\n")

# ADF en Primera Diferencia (d=1)
adf_diff = adfuller(serie_precios.diff().dropna())
print(f"2. Serie Diferenciada (d=1): Estadístico = {adf_diff[0]:.4f} | p-valor = {adf_diff[1]:.4f}")
print("   Veredicto: Se rechaza H0. La serie es ESTACIONARIA.")

--- PRUEBAS DE DICKEY-FULLER AUMENTADA (ADF) ---
1. Serie en Niveles (d=0): Estadístico = -2.8825 | p-valor = 0.0474
   Veredicto: No se rechaza H0. Presencia de Raíz Unitaria (No Estacionaria).

2. Serie Diferenciada (d=1): Estadístico = -16.7902 | p-valor = 0.0000
   Veredicto: Se rechaza H0. La serie es ESTACIONARIA.


## 2. Matriz de Resultados del Grid Search 
Evaluamos de forma sistemática los tres componentes mediante las combinaciones clave reportadas en la matriz de decisión del reporte en LaTeX para calcular su Log-Verosimilitud y el Criterio de Información de Akaike (AIC).

In [7]:
print("-" * 85)
print(f"{'Modelo ARIMA(p,d,q)':<25} | {'Log-Verosimilitud (ln L)':<25} | {'Métrica AIC':<15}")
print("-" * 85)

# Combinaciones analizadas en el documento académico
modelos_a_evaluar = [
    (0, 0, 0),
    (2, 0, 1),
    (0, 1, 0), 
    (1, 1, 0),
    (0, 1, 1),
    (1, 1, 1)
]

dicc_resultados = {}

for orden in modelos_a_evaluar:
    try:
        modelo = ARIMA(serie_precios, order=orden)
        resultado_ajuste = modelo.fit()
        
        log_likelihood = resultado_ajuste.llf
        aic = resultado_ajuste.aic
        
        # Almacenar el objeto de ajuste para la siguiente celda
        dicc_resultados[orden] = resultado_ajuste
        
        nombre_modelo = f"ARIMA{orden}"
        if orden == (0, 1, 0):
            nombre_modelo += " (*)" # Indicador de óptimo
            
        print(f"{nombre_modelo:<25} | {log_likelihood:<25.2f} | {aic:<15.2f}")
    except Exception as e:
        print(f"ARIMA{orden:<20} | Error de Convergencia: {e:<25} | N/A")

print("-" * 85)
print("(*) Modelo seleccionado por minimización global de AIC bajo el principio de parsimonia.")

-------------------------------------------------------------------------------------
Modelo ARIMA(p,d,q)       | Log-Verosimilitud (ln L)  | Métrica AIC    
-------------------------------------------------------------------------------------
ARIMA(0, 0, 0)            | -625.05                   | 1254.10        
ARIMA(2, 0, 1)            | -280.26                   | 570.52         
ARIMA(0, 1, 0) (*)        | -279.17                   | 560.34         
ARIMA(1, 1, 0)            | -278.64                   | 561.29         
ARIMA(0, 1, 1)            | -278.63                   | 561.25         
ARIMA(1, 1, 1)            | -278.08                   | 562.16         
-------------------------------------------------------------------------------------
(*) Modelo seleccionado por minimización global de AIC bajo el principio de parsimonia.


## 3. Diagnóstico y Veredicto Estadístico
Validamos que el modelo seleccionado **ARIMA(0,1,0)** haya extraído toda la estructura lineal analizando sus residuos con la prueba de **Ljung-Box** (comprobación de ruido blanco).

In [8]:
mejor_orden = (0, 1, 0)

if mejor_orden in dicc_resultados:
    modelo_optimo = dicc_resultados[mejor_orden]
    
    # Prueba de Ljung-Box para 10 rezagos (lags)
    prueba_lb = acorr_ljungbox(modelo_optimo.resid, lags=[10], return_df=True)
    p_valor_lb = prueba_lb['lb_pvalue'].values[0]
    
    print(f"Prueba de Ljung-Box sobre los residuos del modelo óptimo (Lag=10):")
    print(f"p-valor calculado = {p_valor_lb:.4f}")
    
    if p_valor_lb > 0.05:
        print("Veredicto: p-valor > 0.05. Se acepta H0: LOS RESIDUOS SON RUIDO BLANCO PURO.")
        print("El modelo es válido y no requiere parámetros adicionales.")
    else:
        print("Alerta: Los residuos muestran autocorrelación remanente.")
        
    print("\n" + "="*78)
    print("                 RESUMEN ECONOMÉTRICO FORMAL (STATSMODELS)")
    print("="*78)
    print(modelo_optimo.summary())
else:
    print("Error: El modelo ARIMA(0,1,0) no fue procesado correctamente.")

Prueba de Ljung-Box sobre los residuos del modelo óptimo (Lag=10):
p-valor calculado = 1.0000
Veredicto: p-valor > 0.05. Se acepta H0: LOS RESIDUOS SON RUIDO BLANCO PURO.
El modelo es válido y no requiere parámetros adicionales.

                 RESUMEN ECONOMÉTRICO FORMAL (STATSMODELS)
                               SARIMAX Results                                
Dep. Variable:                      y   No. Observations:                  250
Model:                 ARIMA(0, 1, 0)   Log Likelihood                -279.169
Date:                Sun, 07 Jun 2026   AIC                            560.338
Time:                        23:25:44   BIC                            563.855
Sample:                             0   HQIC                           561.753
                                - 250                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
